# Computational Astrobiology: Lecture & Tutorial Notes

## Tutorial 1: Bayesian Inference, Regularization, and Gaussian Processes

This module establishes the foundational statistical techniques required to model astronomical data with quantified uncertainties, moving from classical regression to robust probabilistic frameworks.

### Physical Background

The tutorial applies regression models to cosmological data to infer the relationship between distance modulus ($\mu$) and redshift ($z$)—the classic Hubble diagram. It also addresses Total Least Squares, which is critical in astronomy when both independent and dependent variables contain significant observational uncertainties.

### Methodology

* **Bayesian Linear Regression & Likelihoods:** Computes likelihood contours over parameter grids to determine best-fit parameters, factoring in upper/lower bounds mathematically.


* **Regularized Regression (Ridge & Lasso):** Introduces penalties to prevent overfitting when mapping complex relationships using basis functions (like Gaussian kernels).


* **Robust Loss Functions:** Transitions from standard Mean Squared Error to the Huber loss function, minimizing the influence of outliers while maintaining differentiability.


* **Markov Chain Monte Carlo (MCMC):** Utilizes PyMC3 to build mixture models that actively fit and marginalize out nuisance variables (outliers) during regression.


* **Gaussian Processes (GP):** Implements non-parametric regression using squared exponential (Radial Basis Function) kernels, constraining covariance using noisy observations.



**Key Equations:**

* Huber Loss:

$$\Phi(t, c) = \begin{cases} \frac{1}{2}t^2 & \text{if } |t| \le c \\ c(|t| - \frac{1}{2}c) & \text{if } |t| > c \end{cases}$$


* Gaussian Basis/RBF Kernel:

$$K(x_1, x_2) = \exp\left(-\frac{(x_1 - x_2)^2}{2h^2}\right)$$



### Annotated Code Snippets

```python
# Defining the robust Huber Loss for outlier mitigation
def huber_loss(m, b, x, y, dy, c=2):
    y_fit = m * x + b
    t = abs((y - y_fit) / dy)
    flag = t > c
    # Applies quadratic loss to inliers, linear loss to outliers
    return np.sum((~flag) * (0.5 * t ** 2) - (flag) * c * (0.5 * c - t), -1)

# Gaussian Process regression with sci-kit learn
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(10, (1e-2, 1e2))
gp = GaussianProcessRegressor(kernel=kernel, alpha=dmu ** 2)
gp.fit(z_sample[:, None], mu_sample)
y_pred, sigma = gp.predict(z_fit[:, None], return_std=True)

```

### Conclusion

By implementing MCMC mixture models, the regression bounds tightly to the main distribution, fully rejecting extreme outliers automatically. Gaussian Processes map the underlying continuous functions of the $\mu$ vs $z$ cosmology data flawlessly without requiring pre-defined polynomial degrees. Cross-validation strictly proves that higher-degree polynomials overfit the noise in astronomical sets, yielding rapidly escalating Bayesian Information Criterion (BIC) penalties.

---

## Tutorial 2a: Naive Bayes Classification of Exoplanet Discovery Methods

This module introduces generative, probabilistic machine learning using the NASA Exoplanet Archive to classify how planets were discovered.

### Physical Background

The dataset contains confirmed planetary properties (period, radius, mass, density, eccentricity) alongside host star properties (effective temperature, metallicity, surface gravity). The operational label distinguishes exoplanets discovered by the "Transit" method versus all other methods, a split heavily influenced by observational selection biases.

### Methodology

* **Gaussian Naive Bayes:** Assumes features are independent and fit a Gaussian distribution within each class.


* **Data Imputation:** Handles missing catalogue values using median substitution, preserving continuous data streams for the algorithm.


* **Evaluation Metrics:** Assesses performance using Precision, Recall, F1-scores, Confusion Matrices, and Receiver Operating Characteristic (ROC) Area Under the Curve (AUC).



**Key Equations:**

* Gaussian Class-Conditional Probability:

$$P(x_i \mid y) = \frac{1}{\sqrt{2\pi\sigma_y^2}} \exp\left(-\frac{(x_i - \mu_y)^2}{2\sigma_y^2}\right)$$



### Annotated Code Snippets

```python
# Pipeline combining imputation and Naive Bayes
nb_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', GaussianNB()),
])

# Training and evaluating the model
nb_model.fit(X_train, y_train)
y_pred = nb_model.predict(X_test)
y_proba = nb_model.predict_proba(X_test)[:, 1]

# Extracting class-conditional parameters learned by the model
gnb = nb_model.named_steps['clf']
means = pd.DataFrame(gnb.theta_, columns=feature_cols)
variances = pd.DataFrame(gnb.var_, columns=feature_cols)

```

### Conclusion

Log-transforming highly skewed features, such as orbital period and insolation, drastically improves the Gaussian Naive Bayes model by aligning the data with the algorithm's foundational assumption of normality. When Logistic Regression is applied to the same dataset, it outperforms Naive Bayes because it discriminatively optimizes weights and naturally mitigates the strict (and violated) feature-independence assumptions required by Bayes' theorem.

---

## Tutorial 2b: Habitable-Zone Classification with Imbalanced Data

Expanding on classification, this module refines target definition and tackles extreme class imbalances using cross-validation.

### Physical Background

A toy "habitable-zone" proxy is constructed based on stellar insolation bounds (0.25 to 1.75 Earth units) and a planetary radius cutoff ($\le$ 1.8 Earth radii). Real habitability requires complex atmospheric and orbital assessments; thus, this operational label isolates an extremely sparse subset of the total exoplanet catalogue for classification.

### Methodology

* **Target Definition & Leakage Prevention:** Constructs a custom binary label while strictly removing the defining variables (insolation and radius) from the predictive feature matrix to avoid trivial classification.


* **Stratified K-Fold Cross-Validation:** Preserves the highly skewed positive-to-negative class ratio across all training and testing folds.


* **Imbalanced Metrics:** Emphasizes precision, recall, and F1 over raw accuracy, as an algorithm predicting "0" universally would achieve $>99\%$ accuracy.



### Annotated Code Snippets

```python
# Defining the toy habitable-zone label
df_hz['hz_toy'] = ((df_hz['pl_insol'].between(0.25, 1.75)) & (df_hz['pl_rade'] <= 1.8)).astype(int)

# Stratified Cross-Validation on highly imbalanced data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    estimator=nb_model,
    X=X_model,
    y=y,
    cv=cv,
    scoring={'precision': 'precision', 'recall': 'recall', 'f1': 'f1'},
    n_jobs=-1,
)

```

### Conclusion

By entirely excluding radius and insolation from the input, the classifier is forced to rely on orbital mechanics and stellar properties, successfully eradicating target leakage. In this highly imbalanced environment, evaluating with Stratified K-fold and focusing on the F1-score immediately reveals that Naive Bayes produces severe false positives, failing to effectively isolate the sparse positive class without calibrated probabilistic thresholds.

---

## Tutorial 3a: Neural Networks and Gradient Descent

This tutorial derives and implements the core mechanics of Deep Learning, bridging manual backpropagation with modern PyTorch implementations.

### Physical Background

The regression model predicts the planetary equilibrium temperature (`pl_eqt`), an approximation of top-of-atmosphere radiative balance dictated fundamentally by stellar effective temperature, stellar radius, and semi-major axis.

### Methodology

* **Activation Functions & Derivatives:** Compares Sigmoid, Tanh, ReLU, and Leaky ReLU, analyzing their derivatives to understand gradient flow.


* **Loss Geometries:** Evaluates MSE, MAE, and Huber losses, analyzing their sensitivity to outliers in the exoplanet catalogue.


* **Manual Gradient Descent:** Computes gradients via the chain rule and iteratively updates weights on a toy 1D dataset to visualize convergence.


* **PyTorch MLPs:** Constructs a multi-layer perceptron with `torch.nn`, tracking gradient norms explicitly to monitor vanishing/exploding conditions.


* **Binary Cross-Entropy (BCE):** Utilizes BCE for classification, matching logarithmic penalty geometry with binary likelihoods.



**Key Equations:**

* Gradient of Mean Squared Error:

$$\frac{\partial L}{\partial w} = \frac{2}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i) x_i$$


* Binary Cross Entropy:

$$L_{BCE} = -\frac{1}{N} \sum_{i} \left[ y_i \log(\hat{p}_i) + (1 - y_i)\log(1 - \hat{p}_i) \right]$$



### Annotated Code Snippets

```python
# Implementing manual gradients for a single neuron (Chain Rule)
for epoch in range(epochs):
    z = w * x + b
    y_pred = phi(z)
    loss = loss_fn(y, y_pred)
    
    dL_dyhat = loss_grad(y, y_pred)
    dyhat_dz = dphi(z)
    
    dL_dw = np.sum(dL_dyhat * dyhat_dz * x)
    dL_db = np.sum(dL_dyhat * dyhat_dz)
    
    w -= lr * dL_dw
    b -= lr * dL_db

```

### Conclusion

Deploying ReLU and LeakyReLU completely bypasses the gradient saturation intrinsic to Sigmoid activations, yielding drastically faster and more stable loss convergence over multiple epochs. Removing core stellar features (mass, radius, temperature) completely destroys the model's ability to map equilibrium temperatures, confirming the neural network is explicitly learning the underlying physical radiative scaling laws.

---

## Tutorial 3b: Exoplanet Detection using SMOTE and Decision Trees

This tutorial focuses on handling noisy, raw intensity features and extreme class imbalances within time-series classification.

### Physical Background

The dataset contains sequential flux measurements representing light intensity received from stars. Dips in this flux array signal the transit of an exoplanet. The data reflects Kepler space telescope observations, meaning transits are exceptionally rare compared to baseline light curves.

### Methodology

* **Gaussian Filtering:** Applies continuous Gaussian filters (`scipy.ndimage`) to smooth transit noise.


* **SMOTE Oversampling:** Uses the Synthetic Minority Over-sampling Technique to synthetically generate transit signals in the latent space, perfectly balancing the 0 and 1 classes.


* **Supervised Classifiers:** Deploys K-Nearest Neighbors, Logistic Regression, and Decision Trees (`max_depth=5`) on the oversampled flux arrays to detect planetary transits.



### Annotated Code Snippets

```python
# Applying a Gaussian filter to smooth time-series flux arrays
x_train = ndimage.filters.gaussian_filter(x_train, sigma=10)
x_test = ndimage.filters.gaussian_filter(x_test, sigma=10)

# Balancing the severe class imbalance with SMOTE
from imblearn.over_sampling import SMOTE
model = SMOTE()
ov_train_x, ov_train_y = model.fit_resample(train_data.drop('LABEL', axis=1), train_data['LABEL'])

# Training a Decision Tree on the synthetic balanced data
ds_model = DecisionTreeClassifier(max_depth=5, random_state=13)
ds_model.fit(ov_train_x, ov_train_y)

```

### Conclusion

Balancing the heavily skewed dataset via SMOTE forces the classifiers to recognize the actual morphological patterns of the transit curves rather than passively predicting the majority "non-exoplanet" class. Once balanced, the Decision Tree establishes highly accurate decision thresholds across the flux array, isolating transit events with exceptional recall and robust F1 scores.

---

## Tutorial 4: Support Vector Machines & Feature Leakage

This module demonstrates boundary maximization algorithms and explores the critical danger of definition leakage in scientific modeling.

### Physical Background

The tutorial identifies "Hot Jupiters" operationally using strict cuts: Orbital Period $< 10$ days and Planetary Radius $> 8$ Earth radii.

### Methodology

* **Linear & RBF SVMs:** Compares Linear SVC (seeking a hard linear hyperplane) against Radial Basis Function SVC (projecting into infinite-dimensional space to find non-linear boundaries).


* **Feature Leakage Analysis:** Conducts an A/B test. Experiment A trains the SVM with the defining features (Period and Radius) included. Experiment B strictly removes them.


* **Support Vector Analysis:** Identifies the specific planetary systems resting exactly on the decision margin to interpret borderline categorizations.



**Key Equations:**

* Decision Function:

$$f(x) = \mathbf{w}^T \mathbf{x} + b$$



### Annotated Code Snippets

```python
# Pipeline with Linear SVC to identify decision boundaries
linear_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LinearSVC(C=1.0, max_iter=20000, dual=False))
])

# Extracting feature coefficients to detect leakage
coef = linear_pipe.named_steps["clf"].coef_[0]
coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coef})

# Identifying exact border support vectors
sv_idx = svc_support.named_steps["clf"].support_
support_examples = X_train_B.iloc[sv_idx]

```

### Conclusion

Experiment A achieves near-perfect accuracy because the SVM directly exploits the features used to define the label, perfectly illustrating target leakage. Removing period and radius in Experiment B drops accuracy but forces the SVM to leverage physical correlates—primarily the semi-major axis—proving the model inherently recognizes Kepler's Third Law to predict orbital closeness indirectly.

---

## Tutorial 5: Random Forests and Permutation Importance

This module leverages ensemble decision trees to classify physical planetary typologies and safely interpret feature contributions.

### Physical Background

Planets are classified into morphological bins: rocky ($< 1.8 R_\oplus$), sub-Neptune ($1.8 \le R_p < 4.0 R_\oplus$), and giant ($\ge 4.0 R_\oplus$). The classification is driven by intrinsic structural metrics like planetary mass (`pl_bmasse`) and bulk density (`pl_dens`), alongside orbital architectures.

### Methodology

* **Random Forest Classifier:** Combines multiple decision trees via bagging and feature sub-sampling (`max_features="sqrt"`) to map complex, non-linear physical interactions.


* **Permutation Feature Importance:** Randomly shuffles individual feature columns during testing to measure the corresponding drop in balanced accuracy. This is far more reliable for scientific interpretation than native tree-impurity metrics.


* **Class Weighting:** Uses `class_weight="balanced_subsample"` to account for catalog sampling imbalances.



### Annotated Code Snippets

```python
# Random Forest model pipeline
rf = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42
)

# Evaluating feature importance securely via permutation
perm = permutation_importance(
    model, X_test, y_test, n_repeats=10, scoring="balanced_accuracy"
)
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean
})

```

### Conclusion

The permutation importance rigorously confirms that planetary mass and bulk density are the overwhelming drivers of radius classification, directly reflecting bulk compositional physics. By entirely removing these intrinsic structural variables, the model’s balanced accuracy craters, proving that external stellar and environmental features alone are completely insufficient to predict a planet's physical volume.

---

## Tutorial 6: CUDA Programming Fundamentals

Shifting to computational architecture, this tutorial defines the mechanics required to push high-dimensional data computations out of the CPU and onto NVIDIA GPUs.

### Methodology

* **Host vs. Device:** Maps CPU data preparation directly to GPU execution workflows.


* **Hierarchy & Indexing:** Calculates global memory indexes using the execution grid mapping of threads and blocks.


* **Memory Bounds vs Compute Bounds:** Employs the Arithmetic Intensity (FLOPs per byte) model. Element-wise operations saturate bandwidth, whereas Matrix Multiplications (GEMM) leverage shared-memory tiling to become compute-bound.


* **Kernel Fusion:** Combines sequential operations (e.g., Matmul followed by ReLU) into single instructions to eliminate intermediate VRAM read/writes.


* **Softmax & Attention Scalers:** Explains the numerical stability behind normalized softmax and dot-product attention scalers.



**Key Equations:**

* Global Thread Indexing:

$$\text{idx} = \text{blockIdx.x} \times \text{blockDim.x} + \text{threadIdx.x}$$


* Grid Ceiling Division:

$$\text{grid} = \left\lceil \frac{N}{B} \right\rceil = \frac{N + B - 1}{B}$$


* Scaled Dot-Product Attention:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$



### Annotated Code Snippets

```python
# Synchronized benchmarking for accurate GPU timings
def benchmark(fn, iters=50):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    return (t1 - t0) * 1e3

# Numerically stable, CUDA-efficient Softmax
m = x.max()
manual_stable = torch.exp(x - m) / torch.exp(x - m).sum()

```

### Conclusion

Structuring PyTorch operations as contiguous block sequences strictly ensures memory coalescing, heavily reducing memory fetch latencies. Dividing attention query-key dots by the square root of the dimension strictly normalizes the variance, preventing the softmax output from collapsing into extreme binary states and destroying the gradient during backpropagation.

---

## Tutorial 7a: CUDA-Accelerated Spectral Analysis

This tutorial merges the astrophysics of spectroscopy with the high-performance parallel processing mechanics detailed in Tutorial 6.

### Physical Background

Real exoplanet host-star transmission and emission spectra gathered from the MAST archive (JWST/HST) are processed. Parallel programming paradigms perfectly suit 1D and 2D tensor mappings of wavelength windows.

### Methodology

* **Warp Divergence Mitigation:** Uses logical masking (`torch.where`) rather than boolean indexing to prevent branch divergence inside GPU warps across spectral masks.


* **Spectral Reductions:** Computes global constants (maximum flux, L2 norms) using synchronized reduction kernels.


* **Polynomial Design Matrices:** Recasts spectral fitting as a GEMM operation (General Matrix Multiply), exploiting the high arithmetic intensity of GPUs.


* **Self-Attention on Spectra:** Treats localized rolling wavelength windows as individual "tokens" and computes their attention scores relative to one another.



### Annotated Code Snippets

```python
# Utilizing torch.where to avoid warp divergence in CUDA
def branch_free_spectral_mask(flux, wave, lo, hi):
    mask = (wave > lo) & (wave < hi)
    return torch.where(mask, 1.2 * flux, 0.8 * flux)

# Self-Attention implemented manually across spectral window tokens
scores = (Q @ K.transpose(0, 1)) / math.sqrt(D_model)
weights = torch.softmax(scores, dim=-1)
attn_out = weights @ V

```

### Conclusion

Deploying `torch.where` entirely averts SIMT warp divergence, processing the spectral logic masks vastly faster than traditional boolean array indexing. Framing polynomial baseline fits as an explicit matrix multiplication completely shifts the task into the compute-bound regime, maximizing GPU throughput and minimizing low-intensity element-wise memory transfers.

---

## Tutorial 7b: Autoencoders on MUSCLES Host-Star Spectra

This module utilizes Deep Learning for Unsupervised Representation Learning to automatically classify spectral morphologies.

### Physical Background

The dataset consists of Mega-MUSCLES Spectral Energy Distributions (SEDs) from the MAST archive. Integrating these X-ray-to-IR spectra across standard UV bands (UV-A, UV-B, UV-C) provides direct proxies for prebiotic photochemistry and atmospheric mass-loss conditions.

### Methodology

* **Data Resampling:** Standardizes disparate FITS files by interpolating fluxes onto a uniform, logarithmically spaced wavelength grid.


* **Dataset Augmentation:** Generates synthetic variance by injecting Gaussian noise and randomly masking spectral segments to expand the limited training pool.


* **Dense Autoencoders:** Trains a multi-layer Neural Network to compress the high-dimensional SEDs into a compact bottleneck (latent vector), before attempting to decode it back into the original spectrum.


* **Latent Space Mapping:** Applies PCA to the resulting latent vectors and correlates the coordinates via Spearman rank to the derived astrobiological UV proxies.



### Annotated Code Snippets

```python
# Common grid interpolation for multi-instrument spectra
grid_A = np.geomspace(10, 1e5, 2048)
x = np.log10(wl)
y = np.log10(fl)
interp = np.interp(np.log10(grid_A), x, y)

# Pushing spectra through the trained encoder bottleneck
model.eval()
with torch.no_grad():
    X_recon, Z = model(X_tensor)
    
# Extracting the embedded 8-dimensional latent vectors
latent_vectors = Z.cpu().numpy()

```

### Conclusion

The Autoencoder successfully forces the complex, multi-decade spectral data into an 8-dimensional latent representation that autonomously clusters distinct stellar types. Extracting the first principal component of this latent space yields an exceptionally high, near-perfect Spearman correlation with the calculated UV-fraction, proving the unsupervised network organically discovered the underlying spectral hardness metric without labels.

---

## Tutorial 8a: Manual Optimization on Exoplanet Radii

This module cracks open the black-box of machine learning libraries, requiring full mathematical derivation and numpy deployment of the optimization loop.

### Physical Background

The notebook sets up a regression task to map logarithmic planetary mass, orbital period, and host star features directly to the log-radius of the planet.

### Methodology

* **Linear & Logistic Gradients:** Derives and calculates exact analytical gradients for linear predictions (MSE loss) and binary boundary tracking (BCE loss).


* **Finite Difference Checking:** Verifies analytical calculus derivations numerically by tweaking weights by an epsilon and observing the loss delta.


* **Manual Update Loops:** Modifies weight tensors explicitly by subtracting the analytically calculated gradient scaled by the learning rate.


* **Standardization:** Validates how feature scaling equates gradient magnitudes across physical attributes that otherwise vary wildly.



**Key Equations:**

* Logistic Gradient Update:

$$\nabla_{\mathbf{w}} L = \frac{1}{N} X^T (\mathbf{p} - \mathbf{y})$$


* Numerical Gradient Checking:

$$\frac{\partial L}{\partial w_k} \approx \frac{L(w_k + \epsilon) - L(w_k - \epsilon)}{2\epsilon}$$



### Annotated Code Snippets

```python
# Implementing analytical gradients for Logistic Regression
def logistic_gradients(X, y, w, b):
    N = X.shape[0]
    p = sigmoid(X @ w + b)
    residual = p - y
    grad_w = (1 / N) * X.T @ residual
    grad_b = (1 / N) * np.sum(residual)
    return grad_w, grad_b

# Numerical Gradient Checking via finite differences
loss_plus = mse_loss(linear_forward(X_train_s, w_plus, b), y_train_s)
loss_minus = mse_loss(linear_forward(X_train_s, w_minus, b), y_train_s)
numeric_grad = (loss_plus - loss_minus) / (2 * epsilon)

```

### Conclusion

The numerical finite-difference results strictly match the analytical gradient values to extreme precision, confirming the accuracy of the manual chain-rule derivations. Pre-standardizing the physical variables maps the features onto unified scales, entirely averting the jagged, unstable gradient updates that occur when processing parameters spanning orders of magnitude.

---

## Tutorial 8b: Exploratory Data Analysis & Scale Invariance

This tutorial handles mixed dataset pipelines and visualizes inherent catalogue biases.

### Physical Background

Analyzes confirmed planets mapping mass-radius boundaries, highlighting how detection mechanisms (transit vs. radial velocity) severely restrict the sampled parameter space and skew overall distributions.

### Methodology

* **Pipeline Assembly:** Combines `SimpleImputer`, `StandardScaler`, and `OneHotEncoder` via `ColumnTransformer` to handle concurrent numeric and categorical astrometric features seamlessly.


* **Logarithmic Transform Filtering:** Removes strictly non-physical inputs (negative orbital periods/masses) prior to $\log_{10}$ conversions to enforce strictly monotonic scaling.


* **Tree Ensembles:** Deploys Random Forest Regressors and Classifiers to establish non-linear relationships across orbital and observational parameters.



### Annotated Code Snippets

```python
# Preprocessing pipelines for mixed astronomical datasets
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, features_num),
    ("cat", categorical_pipe, features_cat)
])

```

### Conclusion

Converting skewed variables like planetary mass and period into base-10 logarithms linearly separates overlapping distributions, drastically accelerating the Random Forest model's ability to find optimal node splits. Limiting the training set strictly to Transit observations forces the model to heavily misclassify long-period wide-orbit planets, precisely documenting the mathematical danger of selection effects.

---

## Tutorial 9a: Principal Component Analysis (PCA)

This module breaks down eigen-decomposition methodologies to visualize high-dimensional variances and execute noise filtering.

### Physical Background

Provides the mathematical architecture required for spectral and image processing (represented by generic digit arrays and Eigenface datasets) by locating optimal basis functions outside traditional pixel-coordinate mapping.

### Methodology

* **Dimensionality Reduction:** Locates the axes of maximum variance, utilizing orthogonal projection to squash multi-dimensional data clouds into two or three highly interpretable continuous dimensions.


* **Cumulative Explained Variance:** Plots the cumulative ratio of variance captured as component counts increase, establishing hard thresholds for dimension truncation.


* **Inverse Transforms & Filtering:** Transforms inputs into PCA space, zeroes out the lowest variance components, and transforms them back, mathematically stripping unstructured background noise out of the arrays.



**Key Equations:**

* Vector/Image Reconstruction via Principal Components:

$$\text{image} = \mu + \sum_{i=1}^N c_i \cdot \text{basis}_i$$



### Annotated Code Snippets

```python
# Utilizing Randomized SVD for massive arrays
pca = PCA(150, svd_solver='randomized', random_state=42)
components = pca.fit_transform(faces.data)

# Reconstructing arrays after severing lower-tier components (Noise Filtering)
projected = pca.inverse_transform(components)

# Tracking the captured variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

```

### Conclusion

Executing an inverse PCA transform after discarding lower-variance vectors strips out stochastic pixel noise effortlessly, returning a heavily smoothed and purified signal array. Assessing the cumulative explained variance plot dictates exactly how many dimensions to keep, perfectly replacing arbitrary dimension guesses with strict threshold limits.

---

## Tutorial 9b: Variational Autoencoders for Galaxy Morphology

This capstone tutorial builds deep generative networks mapped to probabilistic latent topologies using multi-channel survey imagery.

### Physical Background

The dataset is drawn from the Galaxy10 DECaLS archive, containing color-composite ($g, r, z$ band) galaxy cutouts tagged with morphological descriptors (smooth, merging, barred spirals, disturbed).

### Methodology

* **Convolutional VAE:** Generates feature maps via Strided Convolutions and Batch Normalization. Compresses data into parameters representing a multivariate Gaussian distribution ($\mu$, $\log(\sigma^2)$) rather than deterministic points.


* **Reparameterization Trick:** Differentiates through stochastic nodes by treating the sampled variance as $z = \mu + \sigma \odot \epsilon$, allowing loss gradients to flow backward unhindered.


* **Evidence Lower Bound (ELBO):** Minimizes total loss by summing Reconstruction Error (MSE) against the Kullback-Leibler (KL) Divergence penalty (enforcing prior normal distributions).


* **Anomaly Scoring:** Calculates residual magnitudes between raw inputs and reconstructions to mechanically flag rare cosmic interactions.



**Key Equations:**

* ELBO Loss (Reconstruction + KL Divergence):

$$\mathcal{L} = \frac{1}{N} \sum (x_i - \hat{x}_i)^2 - \frac{\beta}{2} \sum \left( 1 + \log(\sigma_i^2) - \mu_i^2 - \sigma_i^2 \right)$$



### Annotated Code Snippets

```python
# The Reparameterization Trick allowing backpropagation
def reparameterize(self, mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + std * eps

# Computing VAE specific loss (MSE + KL Divergence)
def vae_loss(x, x_hat, mu, logvar, beta=1.0):
    rec = F.mse_loss(x_hat, x, reduction="sum") / x.size(0)
    kl = -0.5 * torch.sum(1.0 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    total = rec + beta * kl
    return total, rec, kl

```

### Conclusion

The VAE successfully isolates visual galaxy topologies, smoothly segregating tightly bound ellipticals from wide merging spirals within the t-SNE latent projections. Utilizing the reconstruction residual explicitly isolates transient image artifacts and chaotic galactic mergers with exceptional accuracy by highlighting data structures that drastically diverge from the established learned Gaussian prior.